# First keyframe (GitHub source + Kaggle GPU)

This notebook runs the tracked `pipeline/kaggle_keyframes.py` script for **shot-01 only**. It uses the approved character reference already stored in this repository. No private reference-photo dataset is needed. The launch cell makes one image-generation request and does not retry automatically.

Before running the launch cell, review the shot prompt and make sure you are ready for a model request. The script records the prompt, parameters, image hash, and model call; it does not store credentials.

## 1. Install pinned Kaggle-session dependencies

Enable Kaggle Internet and select a GPU accelerator. These packages are installed in the temporary Kaggle session only.

In [ ]:
%pip install -q "diffusers==0.35.1" "transformers==4.49.0" "accelerate==1.2.1" "safetensors==0.5.3" "jsonschema==4.23.0" "Pillow==11.1.0"

## 2. Verify GPU

In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=True)

## 3. Confirm Hugging Face model access

The SDXL repository may require accepting its model terms on Hugging Face. Use the existing Kaggle Secret `HF_TOKEN`; never paste the token into notebook code.

In [ ]:
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Grant this notebook access to the Kaggle secret HF_TOKEN.")
print("HF_TOKEN loaded from Kaggle Secrets (value hidden).")

## 4. Generate the approved first shot

The launch cell fetches the latest public `main` branch and runs the tracked script for `shot-01` only. It saves results in a new timestamped folder and leaves previous runs untouched. Running this cell makes one image-generation request.

In [ ]:
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = "https://github.com/hann2004/ai_cinematic_storytelling.git"
REPOSITORY_DIR = Path("/kaggle/working/ai_cinematic_storytelling")
if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / ".git").is_dir():
        raise FileExistsError(f"Path exists but is not a Git clone: {REPOSITORY_DIR}")
    remote_url = subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "remote", "get-url", "origin"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if remote_url != REPOSITORY_URL:
        raise RuntimeError(f"Unexpected Git remote at {REPOSITORY_DIR}: {remote_url}")
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
artifact_dir = Path("/kaggle/working/keyframe_artifacts") / run_id
artifact_dir.mkdir(parents=True, exist_ok=False)
environment = os.environ.copy()
environment["HF_TOKEN"] = hf_token
subprocess.run(
    [
        "python",
        "-m",
        "pipeline.kaggle_keyframes",
        "--shot-id",
        "shot-01",
        "--output-dir",
        str(artifact_dir),
    ],
    cwd=REPOSITORY_DIR,
    env=environment,
    check=True,
)
print("Keyframe code came from GitHub:", REPOSITORY_URL)

## 5. Review and download the result

Review the image before using it. The ZIP contains the image, its metadata, and the one-line model-call record. Do not copy Kaggle secrets into project files.

In [ ]:
from IPython.display import FileLink, display
from PIL import Image
from zipfile import ZIP_DEFLATED, ZipFile

image_path = artifact_dir / "shot-01.png"
display(Image.open(image_path))
download_path = Path.cwd() / f"shot-01-artifacts-{run_id}.zip"
with ZipFile(download_path, "w", ZIP_DEFLATED) as archive:
    for filename in ("shot-01.png", "shot-01.json", "shot-01_run.jsonl"):
        archive.write(artifact_dir / filename, arcname=filename)
display(FileLink(download_path.name))